# Agents (Агенты) — Подробный разбор

## Что такое Agent?

**Agent (агент)** — это LLM, который сам принимает решения:
- Какие действия выполнить
- В каком порядке
- Когда остановиться

### Разница между Chain и Agent:

| Аспект | Chain | Agent |
|--------|-------|-------|
| Порядок действий | Жёсткий, определён заранее | Гибкий, решает LLM |
| Логика | Программист пишет код | LLM принимает решения |
| Java аналогия | Pipeline (Stream API) | Strategy Pattern + AI |
| Пример | `prompt → LLM → parser → output` | LLM выбирает: калькулятор? поиск? API? |

### Архитектура Agent:

```
Вопрос пользователя
     ↓
┌─────────────────────────────────┐
│  LLM (мозг агента)              │
│  Думает: что мне нужно сделать? │
└─────────────────────────────────┘
     ↓
Выбирает инструмент (Tool)
     ↓
┌─────────┬─────────┬─────────┐
│ Tool 1  │ Tool 2  │ Tool 3  │
│ Поиск   │ Калькул.│ API     │
└─────────┴─────────┴─────────┘
     ↓
Результат → обратно в LLM
     ↓
LLM думает: достаточно? или ещё что-то нужно?
     ↓
Ответ пользователю
```

### Java аналогия:

```java
// Chain — это как готовый pipeline:
users.stream()
    .filter(u -> u.isActive())
    .map(User::getName)
    .collect(Collectors.toList());
// Порядок жёстко задан программистом

// Agent — это как AI, который сам выбирает стратегию:
interface Tool {
    String execute(String input);
}

class Agent {
    List<Tool> tools;
    
    String solve(String question) {
        while (!isSolved()) {
            // LLM сам решает какой Tool использовать
            Tool selectedTool = llm.chooseStrategy(question, tools);
            String result = selectedTool.execute(question);
            // LLM решает: достаточно или продолжить?
        }
        return finalAnswer;
    }
}
```

## ReAct Pattern (Reasoning + Acting)

**ReAct** — это паттерн работы агента:
1. **Thought (мысль)** — LLM думает что делать
2. **Action (действие)** — выбирает инструмент
3. **Observation (наблюдение)** — получает результат
4. Повторяет пока не найдёт ответ

### Пример работы:

```
Вопрос: "Сколько будет 25 * 4 + 10?"

Thought: Мне нужно посчитать математику
Action: calculator("25 * 4")
Observation: 100

Thought: Теперь нужно добавить 10
Action: calculator("100 + 10")
Observation: 110

Thought: У меня есть ответ
Final Answer: 110
```

### Java аналогия:

```java
// Как отладка (debugging):
// 1. Thought — думаем где баг
// 2. Action — ставим breakpoint
// 3. Observation — смотрим значение переменных
// 4. Повторяем пока не найдём проблему
```

## Установка библиотек

In [1]:
# Установим нужные библиотеки
!pip install -q langchain langchain-openai langchain-core python-dotenv

In [2]:
import os
from dotenv import load_dotenv

# Загружаем API ключ из .env файла
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

print(f"API Key загружен: {api_key[:10]}..." if api_key else "API Key не найден!")

API Key загружен: sk-***...


## 1. Простой агент с калькулятором

Создадим агента который умеет считать сложные математические выражения.

### Что такое Tool?

**Tool (инструмент)** — это функция Python, которую агент может вызвать.

```python
@tool
def calculator(expression: str) -> str:
    """Полезно для математики"""
    return str(eval(expression))
```

LLM видит:
- Название функции: `calculator`
- Описание: "Полезно для математики"
- Параметры: `expression: str`

И сам решает когда её вызвать!

### Java аналогия:

```java
// Tool в Java — это как @Component в Spring
@Component
public class CalculatorService {
    @Description("Полезно для математики")
    public String calculate(String expression) {
        return evaluator.eval(expression);
    }
}

// А агент — это как ApplicationContext
// Сам находит нужный @Component и вызывает
```

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# Создаём инструмент (Tool)
@tool
def calculator(expression: str) -> str:
    """Полезно когда нужно посчитать математическое выражение.
    Принимает выражение в виде строки, например: '25 * 4 + 10'
    """
    try:
        # eval() выполняет Python код
        # Для production используй безопасные библиотеки!
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Ошибка: {str(e)}"

# Проверим что инструмент работает
print("Тест калькулятора:")
print(calculator.invoke({"expression": "25 * 4 + 10"}))  # Должно быть 110

C:\Users\Ruslan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Тест калькулятора:
110


In [4]:
# Создаём LLM (мозг агента)
llm = ChatOpenAI(
    base_url="https://api.deepseek.com",
    api_key=api_key,
    model="deepseek-chat",
    temperature=0  # 0 = детерминированные ответы
)

# Список инструментов для агента
tools = [calculator]

# Привязываем инструменты к LLM (bind_tools)
llm_with_tools = llm.bind_tools(tools)

print("LLM с инструментами создан!")

LLM с инструментами создан!


In [5]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def run_agent(question: str, max_iterations: int = 5):
    """Простой цикл ReAct агента"""
    messages = [HumanMessage(content=question)]
    
    for i in range(max_iterations):
        print(f"\n{'='*50}")
        print(f"Итерация {i+1}")
        print(f"{'='*50}")
        
        # LLM думает что делать
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        # Если LLM не вызывает инструменты - это финальный ответ
        if not response.tool_calls:
            print("\n✅ ФИНАЛЬНЫЙ ОТВЕТ:")
            print(response.content)
            return response.content
        
        # LLM вызывает инструмент
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            
            print(f"\n🔧 Action: {tool_name}({tool_args})")
            
            # Выполняем инструмент
            if tool_name == "calculator":
                result = calculator.invoke(tool_args)
            else:
                result = f"Неизвестный инструмент: {tool_name}"
            
            print(f"📊 Observation: {result}")
            
            # Добавляем результат в историю
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )
    
    print("\n⚠️ Достигнут лимит итераций")
    return "Не удалось получить ответ"

# Тестируем агента
run_agent("Сколько будет 123 * 456 + 789?")


Итерация 1



🔧 Action: calculator({'expression': '123 * 456 + 789'})
📊 Observation: 56877

Итерация 2



✅ ФИНАЛЬНЫЙ ОТВЕТ:
123 × 456 + 789 = 56 877

Давайте проверим по шагам:
1. 123 × 456 = 56 088
2. 56 088 + 789 = 56 877


'123 × 456 + 789 = 56 877\n\nДавайте проверим по шагам:\n1. 123 × 456 = 56 088\n2. 56 088 + 789 = 56 877'

### Что произошло?

Агент выполнил ReAct цикл:

1. **Thought**: Понял что нужна математика
2. **Action**: Вызвал `calculator("123 * 456 + 789")`
3. **Observation**: Получил результат
4. **Final Answer**: Сформулировал ответ пользователю

### Важно:

- LLM **сам решил** использовать калькулятор
- Мы **не писали** условия `if` для выбора инструмента
- Агент **сам разбил** задачу на шаги

## 2. Агент с несколькими инструментами

Теперь добавим агенту больше возможностей:
- Калькулятор
- Получение текущего времени
- Поиск в списке

Агент сам будет выбирать какой инструмент нужен!

In [6]:
from datetime import datetime

# Инструмент 1: Калькулятор (уже есть)
@tool
def calculator(expression: str) -> str:
    """Полезно для математических вычислений."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Ошибка: {e}"

# Инструмент 2: Текущее время
@tool
def get_current_time() -> str:
    """Возвращает текущее время и дату.
    Не принимает параметров.
    """
    now = datetime.now()
    return now.strftime("%Y-%m-%d %H:%M:%S")

# Инструмент 3: Поиск в базе данных (имитация)
@tool
def search_employee(name: str) -> str:
    """Ищет информацию о сотруднике по имени.
    Принимает имя сотрудника.
    """
    # Имитация базы данных
    employees = {
        "иван": "Иван Петров, Java Developer, 5 лет опыта",
        "мария": "Мария Сидорова, Team Lead, 8 лет опыта",
        "алексей": "Алексей Иванов, DevOps, 3 года опыта"
    }
    
    name_lower = name.lower()
    if name_lower in employees:
        return employees[name_lower]
    else:
        return f"Сотрудник '{name}' не найден"

# Инструмент 4: Конвертер валют (упрощённый)
@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Конвертирует валюту.
    Параметры:
    - amount: сумма
    - from_currency: исходная валюта (USD, EUR, RUB)
    - to_currency: целевая валюта (USD, EUR, RUB)
    """
    # Упрощённые курсы (в реальности используй API)
    rates = {
        "USD_RUB": 90.0,
        "EUR_RUB": 100.0,
        "USD_EUR": 0.9,
        "EUR_USD": 1.11,
        "RUB_USD": 1/90.0,
        "RUB_EUR": 1/100.0
    }
    
    from_currency = from_currency.upper()
    to_currency = to_currency.upper()
    
    if from_currency == to_currency:
        return str(amount)
    
    rate_key = f"{from_currency}_{to_currency}"
    if rate_key in rates:
        result = amount * rates[rate_key]
        return f"{amount} {from_currency} = {result:.2f} {to_currency}"
    else:
        return "Неподдерживаемая пара валют"

print("Инструменты созданы:")
print("1. calculator - математика")
print("2. get_current_time - текущее время")
print("3. search_employee - поиск сотрудника")
print("4. convert_currency - конвертация валют")

Инструменты созданы:
1. calculator - математика
2. get_current_time - текущее время
3. search_employee - поиск сотрудника
4. convert_currency - конвертация валют


In [7]:
# Создаём агента с 4 инструментами
tools = [calculator, get_current_time, search_employee, convert_currency]
llm_with_tools = llm.bind_tools(tools)

# Улучшенная функция run_agent
def run_multi_tool_agent(question: str, max_iterations: int = 10):
    """Агент с несколькими инструментами"""
    messages = [HumanMessage(content=question)]
    
    # Словарь инструментов для быстрого доступа
    tools_dict = {tool.name: tool for tool in tools}
    
    for i in range(max_iterations):
        print(f"\n{'='*50}")
        print(f"Итерация {i+1}")
        print(f"{'='*50}")
        
        # LLM думает что делать
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        # Если LLM не вызывает инструменты - это финальный ответ
        if not response.tool_calls:
            print("\n✅ ФИНАЛЬНЫЙ ОТВЕТ:")
            print(response.content)
            return response.content
        
        # LLM вызывает инструменты
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            
            print(f"\n🔧 Action: {tool_name}({tool_args})")
            
            # Выполняем инструмент
            if tool_name in tools_dict:
                result = tools_dict[tool_name].invoke(tool_args)
            else:
                result = f"Неизвестный инструмент: {tool_name}"
            
            print(f"📊 Observation: {result}")
            
            # Добавляем результат в историю
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )
    
    print("\n⚠️ Достигнут лимит итераций")
    return "Не удалось получить ответ"

print("Агент с 4 инструментами готов!")

Агент с 4 инструментами готов!


In [8]:
# Тест 1: Просто математика
run_multi_tool_agent("Сколько будет 15 * 8?")


Итерация 1



🔧 Action: calculator({'expression': '15 * 8'})
📊 Observation: 120

Итерация 2



✅ ФИНАЛЬНЫЙ ОТВЕТ:
15 * 8 = 120


'15 * 8 = 120'

In [9]:
# Тест 2: Время
run_multi_tool_agent("Который час?")


Итерация 1



🔧 Action: get_current_time({})
📊 Observation: 2026-01-29 22:02:14

Итерация 2



✅ ФИНАЛЬНЫЙ ОТВЕТ:
Сейчас 22:02 29 января 2026 года.


'Сейчас 22:02 29 января 2026 года.'

In [10]:
# Тест 3: Поиск сотрудника
run_multi_tool_agent("Найди информацию о сотруднике Мария")


Итерация 1



🔧 Action: search_employee({'name': 'Мария'})
📊 Observation: Мария Сидорова, Team Lead, 8 лет опыта

Итерация 2



✅ ФИНАЛЬНЫЙ ОТВЕТ:
Найдена информация о сотруднике:

**Мария Сидорова**
- Должность: Team Lead
- Опыт работы: 8 лет

Это вся доступная информация о сотруднице Мария в системе.


'Найдена информация о сотруднике:\n\n**Мария Сидорова**\n- Должность: Team Lead\n- Опыт работы: 8 лет\n\nЭто вся доступная информация о сотруднице Мария в системе.'

In [11]:
# Тест 4: Конвертация валют
run_multi_tool_agent("Сконвертируй 100 долларов в рубли")


Итерация 1



🔧 Action: convert_currency({'amount': 100, 'from_currency': 'USD', 'to_currency': 'RUB'})
📊 Observation: 100.0 USD = 9000.00 RUB

Итерация 2



✅ ФИНАЛЬНЫЙ ОТВЕТ:
100 долларов США равны **9000 рублей** по текущему курсу.


'100 долларов США равны **9000 рублей** по текущему курсу.'

In [12]:
# Тест 5: Комбинация инструментов!
run_multi_tool_agent("""Найди сколько лет опыта у Марии и Алексея.
Потом посчитай их суммарный опыт.""")


Итерация 1



🔧 Action: search_employee({'name': 'Мария'})
📊 Observation: Мария Сидорова, Team Lead, 8 лет опыта

Итерация 2



🔧 Action: search_employee({'name': 'Алексей'})
📊 Observation: Алексей Иванов, DevOps, 3 года опыта

Итерация 3



🔧 Action: calculator({'expression': '8 + 3'})
📊 Observation: 11

Итерация 4



✅ ФИНАЛЬНЫЙ ОТВЕТ:
**Результаты:**

1. **Опыт Марии**: 8 лет
2. **Опыт Алексея**: 3 года
3. **Суммарный опыт**: 11 лет

Таким образом, Мария имеет 8 лет опыта, Алексей - 3 года опыта, а их общий суммарный опыт составляет 11 лет.


'**Результаты:**\n\n1. **Опыт Марии**: 8 лет\n2. **Опыт Алексея**: 3 года\n3. **Суммарный опыт**: 11 лет\n\nТаким образом, Мария имеет 8 лет опыта, Алексей - 3 года опыта, а их общий суммарный опыт составляет 11 лет.'

### Что произошло в последнем тесте?

Агент **сам составил план**:

1. **Thought**: Нужно найти двух сотрудников
2. **Action**: `search_employee("Мария")`
3. **Observation**: 8 лет опыта
4. **Thought**: Теперь ищу второго
5. **Action**: `search_employee("Алексей")`
6. **Observation**: 3 года опыта
7. **Thought**: Нужно сложить
8. **Action**: `calculator("8 + 3")`
9. **Observation**: 11
10. **Final Answer**: Суммарный опыт 11 лет

Мы **не программировали** эту логику! LLM **сам** придумал план.

## Проблемы и решения

### Проблема 1: Агент зацикливается

**Решение**: `max_iterations=5` — ограничить количество шагов

```python
run_agent(question, max_iterations=5)
```

### Проблема 2: Агент выбирает неправильный инструмент

**Решение**: Улучшить описание инструмента

```python
# Плохо ❌
@tool
def search(query: str) -> str:
    """Поиск"""
    
# Хорошо ✅
@tool
def search_employee(name: str) -> str:
    """Ищет информацию о сотруднике в базе данных.
    
    Используй когда нужно:
    - Найти сотрудника по имени
    - Узнать опыт работы
    - Получить должность
    
    НЕ используй для:
    - Математики (используй calculator)
    - Поиска в интернете
    """
```

### Проблема 3: Дорого (много токенов)

Каждый шаг агента = новый запрос к LLM

**Решение**:
- Использовать более быстрые модели (DeepSeek дёшево!)
- Ограничить `max_iterations`
- Кэшировать повторяющиеся запросы

### Проблема 4: Агент "галлюцинирует" результат

LLM может придумать ответ вместо использования инструмента.

**Решение**: Добавить в system prompt:
```
ВСЕГДА используй инструменты когда они доступны.
НЕ придумывай результаты - только используй инструменты.
```

## Когда использовать Agent vs Chain?

| Используй Chain | Используй Agent |
|-----------------|------------------|
| Задача простая и последовательная | Задача требует принятия решений |
| Порядок шагов всегда одинаковый | Порядок зависит от ситуации |
| Нужна предсказуемость | Нужна гибкость |
| Важна скорость | Важна точность |
| Пример: Generate SQL → Execute → Format | Пример: Анализ → Выбор источника → Поиск → Агрегация |

### Java аналогия:

```java
// Chain — это как Pipeline
data.stream()
    .filter()
    .map()
    .collect();
// Всегда одна и та же последовательность

// Agent — это как Strategy + Factory
Strategy strategy = strategyFactory.choose(context);
Result result = strategy.execute();
while (!isDone) {
    strategy = strategyFactory.chooseNext(result);
    result = strategy.execute();
}
// Выбор стратегии динамический
```

## Итоги

### Что такое Agent:
- LLM который **сам принимает решения**
- Выбирает **какие инструменты** использовать
- Работает по паттерну **ReAct** (Thought → Action → Observation)

### Компоненты Agent:
1. **LLM** — мозг агента
2. **Tools** — инструменты (функции Python с декоратором @tool)
3. **bind_tools()** — привязка инструментов к LLM
4. **ReAct цикл** — наш цикл с HumanMessage/AIMessage/ToolMessage

### Ключевые концепции:
- **@tool** = функция с описанием (LLM видит описание и решает когда вызвать)
- **bind_tools()** = привязка инструментов к LLM
- **tool_calls** = LLM возвращает какие инструменты вызвать
- **ToolMessage** = результат выполнения инструмента

### Java аналогии:
- Tool ≈ `@Component` в Spring
- bind_tools() ≈ `@Autowired` для AI
- ReAct цикл ≈ Debugging (breakpoint → inspect → continue)

### Когда использовать:
- ✅ Задача требует динамических решений
- ✅ Нужно комбинировать разные источники данных
- ✅ Порядок действий зависит от ситуации
- ❌ Простая последовательная задача → используй Chain

### Следующие шаги:
1. **LangGraph** — готовая библиотека для сложных агентов
2. **Custom Tools** — интеграция с API твоей компании
3. **Production** — обработка ошибок, мониторинг, кэширование